In [25]:
import tmu
from PyHierarchicalTsetlinMachineCUDA.tm import TsetlinMachine
import PyHierarchicalTsetlinMachineCUDA.tm as hier_tm_objects

import numpy as np
import pandas as pd
from chembl_structure_pipeline.standardizer import standardize_mol as chembl_standardizer
from rdkit import Chem
from rdkit.Chem import DataStructs, rdFingerprintGenerator

from kennard_stone import train_test_split
from sklearn.metrics import (
    auc,
    mean_absolute_error,
    precision_recall_curve,
    precision_score,
    roc_auc_score,
    roc_curve,
    root_mean_squared_error,
)


In [4]:

def mol_from_smiles(smiles, *args, standardizer=chembl_standardizer) -> Chem.Mol:
    """Convert a SMILES string to an RDKit Mol object.

    :param smiles: SMILES string. Required.
    :param standardizer: function to standardize the molecule. Defaults to
        ChEMBL standardizer. Use None to skip.
    :return: RDKit Mol object, or None if the SMILES could not be parsed.
    """
    mol = Chem.MolFromSmiles(str(smiles))
    if not mol:
        return None
    if standardizer is None:
        return mol
    return standardizer(mol, check_exclusion=True, sanitize=True)


def fp_to_np(fp):
    arr = np.zeros((1,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr


def gen_ecfp_arr(mol_df, mol_col, fp_size=1024, fp_radius=2, n_threads=-1):
    """Morgan (ECFP)-style fingerprints as a binary uint32 array."""
    mols = list(mol_df[mol_col])
    fpg = rdFingerprintGenerator.GetMorganGenerator(radius=fp_radius, fpSize=fp_size)
    fps = fpg.GetFingerprints(mols, numThreads=n_threads)
    fps_np = np.array([fp_to_np(i) for i in fps], dtype=np.uint32)
    return fps_np

In [7]:
def prc_auc_score(Y, Y_pred):
    """Area under the precision-recall curve."""
    precision, recall, _ = precision_recall_curve(Y, Y_pred)
    return auc(recall, precision)

def ppv_npv_score(Y, Y_pred):
    """Positive/negative predictive value at the ROC-optimal (Youden's J)
    probability cutoff."""
    fpr, tpr, proba = roc_curve(Y, Y_pred)
    optimal_proba_cutoff = sorted(
        list(zip(np.abs(tpr - fpr), proba)), key=lambda i: i[0], reverse=True
    )[0][1]

    hard_Y_pred = [1 if p > optimal_proba_cutoff else 0 for p in Y_pred]

    return (
        precision_score(Y, hard_Y_pred, pos_label=1),
        precision_score(Y, hard_Y_pred, pos_label=0),
    )

In [ ]:
mor_data = pd.read_csv("data/MOR.csv")
# Calculate descriptors for the MOR dataset
mor_data["mol"] = mor_data["SMILES"].apply(lambda x: Chem.MolFromSmiles(x))
X = gen_ecfp_arr(mor_data, "mol")


In [17]:
X_train_val, X_test, y_train_val, y_test = train_test_split(X, mor_data["label"], test_size=0.2, device="cuda")

# equal amounts train and test sets
X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.2, device="cuda")

2026-09-27 12:55:46,268 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-09-27 12:55:46,422 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-09-27 12:55:47,307 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-09-27 12:55:47,387 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



In [26]:
# one shot

n_clauses = 10
T = 5000
s = 10
weighted_clauses = True
n_concepts = 5

tm = TsetlinMachine(
    number_of_clauses=n_clauses,
    T=T,
    s=s,
    weighted_clauses=weighted_clauses,
    hierarchy_structure = (
        (hier_tm_objects.AND_GROUP, 1024),
        (hier_tm_objects.OR_ALTERNATIVES, 64),
        (hier_tm_objects.AND_ALTERNATIVES, n_concepts)
    )
)


AttributeError: module 'PyHierarchicalTsetlinMachineCUDA.tm' has no attribute 'AND_ALTERNATIVES'

In [24]:
help(TsetlinMachine)

Help on class TsetlinMachine in module PyHierarchicalTsetlinMachineCUDA.tm:

class TsetlinMachine(CommonTsetlinMachine)
 |  TsetlinMachine(
 |      number_of_clauses,
 |      T,
 |      s,
 |      q=1.0,
 |      log_scale=False,
 |      weighted_clauses=False,
 |      hierarchy_structure=(' ∧ ', 1),
 |      boost_true_positive_feedback=1,
 |      number_of_state_bits=8,
 |      append_negated=True,
 |      grid=(208, 1, 1),
 |      block=(128, 1, 1),
 |      seed=None
 |  )
 |
 |  Method resolution order:
 |      TsetlinMachine
 |      CommonTsetlinMachine
 |      builtins.object
 |
 |  Methods defined here:
 |
 |  __init__(
 |      self,
 |      number_of_clauses,
 |      T,
 |      s,
 |      q=1.0,
 |      log_scale=False,
 |      weighted_clauses=False,
 |      hierarchy_structure=(' ∧ ', 1),
 |      boost_true_positive_feedback=1,
 |      number_of_state_bits=8,
 |      append_negated=True,
 |      grid=(208, 1, 1),
 |      block=(128, 1, 1),
 |      seed=None
 |  )
 |      Initia